# 2. Baseline Strategy

Baseline: 
- Shallow CNN (2 conv layer, pooling layer, flattening layer, fully connected)
- Optimizer: SGD

Improvement: 
- Slightly deeper CNN with batch normalization (5 conv, pooling, flattening, connected)
- Optimizer: adamax
- training epoch up


Using pytorch:
- fix seed, gpu setup
- prep data (train/test, class definition, create dataset, data loader gen)
- Model (CNN)
- Model train (loss, optimizer)
- Eval
- Pred and submit

In [2]:
import torch, random, os
import numpy as np

# Seed fixing
seed = 50
os.environ['PYTHONHASHSEED'] = str(seed)
random.seed(seed)
torch.manual_seed(seed)         # using CPU
torch.cuda.manual_seed(seed)    # using GPU
torch.cuda.manual_seed_all(seed) # multi GPU

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.enabled = False

In [3]:
# Setting up GPU

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

device(type='cpu')

## Preparing dataset

In [5]:
import pandas as pd

data_path = '/kaggle/input/aerial-cactus-identification/'
labels = pd.read_csv(data_path + 'train.csv')
submission = pd.read_csv(data_path + 'sample_submission.csv')


In [6]:
# extract images
from zipfile import ZipFile

with ZipFile(data_path + 'train.zip') as zipper:
    zipper.extractall()

with ZipFile(data_path + 'test.zip') as zipper:
    zipper.extractall()

In [7]:
from sklearn.model_selection import train_test_split

train, valid = train_test_split(labels,
                                test_size=0.1,
                                stratify = labels['has_cactus'],
                                random_state = 50)

print("num train data: ", len(train))
print("num valid data: ", len(valid))

num train data:  15750
num valid data:  1750


## Defining user dataset

In [8]:
import cv2
from torch.utils.data import Dataset

In [9]:
class ImageDataset(Dataset):

    def __init__(self, df, img_dir = './', transform=None):
        super().__init__() # Call inhereted dataset generator
        self.df = df
        self.img_dir = img_dir
        self.transform = transform
    
    def __len__(self): # redefine dataset size
        return len(self.df)

    def __getitem__(self,idx):  # return data on designated index
        img_id = self.df.iloc[idx, 0]    # image ID
        img_path = self.img_dir + img_id # image path
        image = cv2.imread(img_path)     
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB) # fix color
        label = self.df.iloc[idx, 1]    # get label

        if self.transform is not None:
            image = self.transform(image)
        
        return image, label



## Generating dataset using ImageDataset class

In [10]:
# need to convert image to tensor

from torchvision import transforms
transform  = transforms.ToTensor()

# w_pixel, h_pixel, channel -> channel, w_pixel, h_pixel
# final : 3 x 32 x 32



In [11]:
dataset_train = ImageDataset(df=train, img_dir = 'train/', transform=transform)
dataset_valid = ImageDataset(df=valid, img_dir = 'train/', transform=transform)

In [12]:
# For data loader multi processing
def seed_worker(worker_id):
    # def seed fixing func, fix generator seed value
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed)
    random.seed(worker_seed)

g = torch.Generator()
g.manual_seed(0)

    

In [13]:
# Data loader --> load data per defined batch size
from torch.utils.data import DataLoader
loader_train = DataLoader(dataset = dataset_train, batch_size = 32, shuffle=True,    # shuffle for training
                        #   worker_init_fn = seed_worker,
                        #   generator = g,
                        #   num_workers = 2
                          ) 
loader_valid = DataLoader(dataset = dataset_valid, batch_size = 32, shuffle=False,
                        #   worker_init_fn = seed_worker, 
                        #   generator = g,
                        #   num_workers = 2
                          )

# set batch size as power of 2.. per Ibrahem Kandel paper (2020)

## Model Generation
CNN strucure:
Data 3 x 32 x 32

data -> conv nn -> max pool -> conv nn -> max pool -> avg pool -> flatten -> linear layer -> output
                   32x17x17               64x9x9.              64x4x4.   1x1024.                1x2 

In [14]:
import torch.nn as nn
import torch.nn.functional as F 

class Model(nn.Module):
    # Define neural network layers
    def __init__(self):
        super().__init__() # Call nn.Module constructor

        # Setup convolution layers
        self.conv1 = nn.Conv2d(in_channels=3, out_channels=32, 
                               kernel_size=3, padding=2) 
        self.conv2 = nn.Conv2d(in_channels=32, out_channels=64, 
                               kernel_size=3, padding=2) 

        # Setup pooling layers
        self.max_pool = nn.MaxPool2d(kernel_size=2) 
        self.avg_pool = nn.AvgPool2d(kernel_size=2) 

        # Fullly connected layer
        self.fc = nn.Linear(in_features = 64 * 4 * 4, out_features = 2)
    
    # Defining forward prop output
    def forward(self, x):
        x = self.max_pool(F.relu(self.conv1(x)))
        x = self.max_pool(F.relu(self.conv2(x)))
        x = self.avg_pool(x)
        x = x.view(-1, 64 * 4 * 4)  # flattening
        x = self.fc(x)
        return x

Formula for outputsize
$$\text{Output size} = \left\lfloor \frac{H_{\text{in}} + 2P - K}{S} \right\rfloor + 1$$

In [15]:
model = Model().to(device)

model


Model(
  (conv1): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(2, 2))
  (conv2): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(2, 2))
  (max_pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (avg_pool): AvgPool2d(kernel_size=2, stride=2, padding=0)
  (fc): Linear(in_features=1024, out_features=2, bias=True)
)

## Training Model

1. batch load data
2. move image data and label to device
3. initialize gradient in optimizer
4. feed input images into CNN and perform forward pass
5. Compare predictions with true labels and compute loss
6. Perform backprop based on loss
7. update weights using gradient from backprop
8. repeat steps 1-7 for iterations #
9. repeat 1-8 for epochs #



In [16]:
# loss function
criterion = nn.CrossEntropyLoss()

# optimizer
optimizer = torch.optim.SGD(model.parameters(), lr = 0.01) # lr learning rate


In [17]:
# epoch, iteration, batch

import math

math.ceil(len(train) / 32) #iteration
# len(loader_train)



493

In [18]:
epochs = 10 # total number of epochs

# Repeat for the total number of epochs
for epoch in range(epochs):
    epoch_loss = 0 # initialize loss value for each epoch

    # Repeat for the number of iterations (batches)
    for images, labels in loader_train:
        # Move image and label mini-batch data to device
        images = images.to(device)
        labels = labels.to(device)
        
        # Reset gradient in optimizer
        optimizer.zero_grad()
        
        # Forward pass: use image as input
        outputs = model(images)
        
        # Compute loss between outputs and labels using loss func
        loss = criterion(outputs, labels)

        # Accumulate loss for curr batch
        epoch_loss += loss.item() 

        # Backprop
        loss.backward()

        # Update weights
        optimizer.step()
        
    # Print training loss for the epoch
    print(f'Epoch [{epoch+1}/{epochs}] - Loss: {epoch_loss/len(loader_train):.4f}')

KeyboardInterrupt: 